## 線形回帰の例

簡単な例で線形回帰をおこないます。

1. データ収集として説明変数と目的変数を作成します。
2. データ加工としてデータ規格化を行います。
3. データからの学習として

    1. 訓練データとテストデータとに分離します。
    2. 線形回帰を行います。
    3. 回帰後に回帰能を計算しています。

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 100)

In [ ]:
g_data_name = "x5_sin" # x5_sin or x123
g_normalizationtype = "standard" # standard, minmax
g_regtype = "lasso" # linear, lasso, ridge
g_random_state = 1 # random state of train_test_split
g_test_size = 0.25 # test size 

In [ ]:
# 結果保存用のdictを作る。
g_result ={}

## データ取得

データファイルを読み込みます。


ファイル名と、ファイル名内の
説明変数
descriptor_names
と目的変数
target_name
を定義します。

In [ ]:
def get_data(data_name):
    """観測データの作成

    Args:
        data_name (str): 作成するデータの名前。

    Raises:
        ValueError: 規定外のdata_name。

    Returns:
        pd.DataFrame: 観測データ。
        pd.DataFrame: 新規データ
        List(str): 説明変数名のリスト
        str: 目的変数
    """
    if data_name == "x5_sin":
        filename = "../data_calculated/x5_sin.csv"
        filename_new = "../data_calculated/x5_sin_new.csv"
        descriptor_names = ['x1', 'x2', 'x3', 'x4', 'x5', 'x6']
        # descriptor_names = ['x1', 'x2', 'x3', 'x4', 'x5', ]
        target_name = 'y'
    elif data_name == "x123":
        filename = "../data_calculated/x123.csv"
        filename_new = "../data_calculated/x123_new.csv"
        descriptor_names = ['x1', 'x2', 'x3']
        target_name = 'y'
    else:
        raise ValueError("unknown data_name={}".format(data_name))
    df_obs = pd.read_csv(filename)
    df_new = pd.read_csv(filename_new)
    return df_obs, df_new, descriptor_names, target_name

g_df_obs, g_df_new, g_descriptor_names, g_target_name = get_data(g_data_name)

このファイルは以下の内容を持ちます。
'!'から始まるとjupyter上でshell実行します。 

一行目がカラム名、二行目以降がダータインスタンスです。

In [ ]:
!head -n 3 ../data_calculated/x5_sin.csv

In [ ]:
g_descriptor_names

In [ ]:
g_target_name

In [ ]:
g_df_obs

In [ ]:
g_df_new

In [ ]:
g_df = pd.concat([g_df_obs, g_df_new], axis=0).reset_index(drop=True)
g_df.plot(x="x1", y="y")

DataFrameからdescriptor_names,target_nameを切り出しnumpyに直し、
説明変数Xrawと目的変数yの生成を行う。

In [ ]:
# obs
g_Xraw = g_df_obs.loc[:, g_descriptor_names].values
g_y = g_df_obs.loc[:, g_target_name].values
# new
g_Xraw_new = g_df_new.loc[:, g_descriptor_names].values
g_y_new = g_df_new.loc[:, g_target_name].values

### データ加工：プリプロセス
観測生データ＝加工済み観測データができているので、ここでは規格化のみ行います。

In [ ]:
def scale_X(Xraw, normalizationtype=None, scaler=None):
    """Xを規格化する。

    Args:
        Xraw (np.ndarray): 説明変数。
        normalizationtype (str, optional): 規格化の名前. Defaults to None.
        scaler (StandardScaler|MinMaxScaler, optional): 規格化クラスインスタンス. Defaults to None.

    Raises:
        ValueError: 規定外normalizationtype

    Returns:
        nd.ndarray: 規格化された説明変数

    """
    if scaler is not None:
        print("use", scaler)
        X = scaler.transform(Xraw)
    else:
        print("normalizationtype", normalizationtype)
        if normalizationtype=="standard":
            from sklearn.preprocessing import StandardScaler
            scaler = StandardScaler()
            scaler.fit(Xraw)
            X = scaler.transform(Xraw)    
        elif normalizationtype=="minmax":
            from sklearn.preprocessing import MinMaxScaler
            scaler = MinMaxScaler()
            scaler.fit(Xraw)
            X = scaler.transform(Xraw)    
        elif normalizationtype is None:
            # 規格化を行わない。
            X = Xraw
            scaler = None
        else:
            raise ValueError("unkown normalizationtype={}".format(normalizationtype))
    return X, scaler

g_X, g_scaler = scale_X(g_Xraw, g_normalizationtype)
g_X_new, _ = scale_X(g_Xraw_new, scaler=g_scaler)

In [ ]:
plt.plot(g_X)

In [ ]:
plt.plot(g_X_new)

### データからの学習

1. モデル当てはめと、その性能評価と
2. 予測

を行います。

1. まずモデル当てはめの良さ(観測データ全てでfit()をしてpredict()して、性能指標）を評価します。

In [ ]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge

def choose_linear_model(regtype :str, alpha:float=1e-2):
    """線形モデルの選択を行う

    Args:
        regtype (str): 線形モデル名
        alpha (float, optional): Lasso, Ridgeのhyperparameter. Defaults to 1e-2.

    Raises:
        ValueError: 規定外線形モデル名。

    Returns:
        LinearRegression|Lasso|Ridge: 線型回帰モデルinstance
    """
    print("alpha", alpha)
    if regtype=="linear":
        reg = LinearRegression()
    elif regtype=="lasso":
        reg = Lasso(alpha=alpha)
    elif regtype=="ridge":
        reg = Ridge(alpha=alpha)
    else:
        raise ValueError("unkown regtype={}".format(regtype))
    return reg

def predict_and_score(X,y, regtype, reg=None, alpha=1e-2):
    """回帰モデルをfitしてr2scoreを出す。

    Args:
        X (np.ndarray)): 説明変数
        y (np.ndarray)): 目的変数
        regtype (str): 線型回帰モデル名
        reg (LinearRegression|Lasso|Ridge, optional): 線型回帰モデル. Defaults to None.

    Returns:
        np.ndarray: 予測値
        LinearRegression|Lasso|Ridge: 線型回帰モデル
    """
    if reg is None:
        reg = choose_linear_model(regtype, alpha)
        reg.fit(X,y)

        # fitで回帰係数が求まっている。
        print(reg.coef_, reg.intercept_)

    # R2 の計算を行う。
    all_r2_score = reg.score(X,y)
    print("R2",all_r2_score)
    return y, reg

g_y, g_reg = predict_and_score(g_X, g_y, g_regtype)
g_y_new, _ =  predict_and_score(g_X_new, g_y_new, g_regtype, g_reg)

RMSE,MAE,R2を$y^{pred}$から評価する。
R2は同じ値になる。

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
def get_all_scores(y,yp):
    """ RMSE, MAE, R2を計算する。

    Args:
        y (np.ndarray): 目的変数観測値
        yp (np.ndarray): 目的変数予測値

    Returns:
        dict: RMSE, MAE, R2
    """
    rmse = np.sqrt(mean_squared_error(y,yp))
    mae = mean_absolute_error(y,yp)
    r2 = r2_score(y,yp) 
    print("rmse",rmse,"mae",mae, "R2",r2)
    return  {"rmse":rmse, "mae":mae, "R2": r2}

# 後で使うので結果の保存を行う。
g_yp = g_reg.predict(g_X)
g_result["all"] = get_all_scores(g_y,g_yp)

回帰係数の表示を行う。

### 可視化

$y^{obs}$ vs $y^{pred}$の図示を行う。

In [ ]:
def plot_y_yp(y,yp, title: str):
    """y vs ypを図示する。

    Args:
        y (np.ndarray): 目的変数観測値
        yp (np.ndarray): 目的変数予測値 
        title (str): 図のtitle
    """
    fig, ax = plt.subplots(figsize=(5,5))

    # $y^{obs}$ vs $y^{predict}$
    ax.plot(y,yp,"o")

    # 斜め線を引く
    yall = np.hstack([y,yp])
    ylim = yall.min(), yall.max()
    ax.plot(ylim,ylim,"--")

    # labelを書く
    ax.set_xlabel("$y_{obs}$")
    ax.set_ylabel("$y_{pred}$")
    ax.set_title(title)
    fig.show()
    
plot_y_yp(g_y, g_yp, "all")

### 予測性能の評価

#### 訓練データとテストデータへの分離による評価

テストデータ数が観測データの25%になるように分離する。
train_test_splitのshuffleはdefaultでTrueであるが、念のため明示する。
（version upでdefault値は変わる可能性がある。）

In [ ]:
from sklearn.model_selection import train_test_split
g_Xtrain, g_Xtest, g_ytrain, g_ytest = train_test_split(g_X,g_y,test_size=g_test_size, 
                                                shuffle=True, random_state=g_random_state) 

訓練データで回帰モデルをfit()して回帰係数を表示する。

In [ ]:
g_regtype

In [ ]:
g_reg = choose_linear_model(g_regtype)
g_reg.fit(g_Xtrain, g_ytrain)
# 回帰係数と切片を表示する。
print(g_reg.coef_, g_reg.intercept_)


テストデータで$R^2$を評価する。

In [ ]:
g_test_r2_score = g_reg.score(g_Xtest,g_ytest)
print("R2(test)",g_test_r2_score)

訓練データに対する予測値（ytrainp)とテストデータに対する予測値(ytestp)の生成を行う。

R2以外の性能指標を評価する。

In [ ]:
g_ytrainp = g_reg.predict(g_Xtrain)
g_ytestp = g_reg.predict(g_Xtest)
print("using training data")
g_result["R2(train)"] = get_all_scores(g_ytrain,g_ytrainp)
print("using test data")
g_result["R2(test)"] = get_all_scores(g_ytest,g_ytestp)

resultに保存したall, test, trainの結果を比較する。

- all: 観測データ全てで回帰モデルを学習して予測した性能指標値
- train: 訓練データで回帰モデルを学習して訓練データに対して予測した性能指標値
- test: 訓練データで回帰モデルを学習してテストデータに対して予測した性能指標値

です。trainはallの一部のデータインスタンスを用いているという違いしか無い。

In [ ]:
g_result

In [ ]:
pd.DataFrame(g_result)


$\textrm{RMSE}^2_{all}\simeq \textrm{RMSE}^2_{train} < \textrm{RMSE}^2_{test}$、
$R^2_{all}\simeq R^2_{train} > R^2_{test}$
という関係になって欲しいですが、
データインスタンス数が少ないので、
random_stateに値が大きく依存します。


#### 新規データに対する予測

In [ ]:
g_yp_new = g_reg.predict(g_X_new)

# 可視化は後で行う。

### データ可視化

データプリプロセスにより
規格化された説明変数X(Xrawは規格化していない）は値の範囲がほぼ同じことを可視化して確認します。

訓練データ、テストデータを表示する。

Xは規格化しているのでX0の範囲はXrawの場合と異なります。

In [ ]:
def plotX(Xtrain,Xtest,X):
    """説明変数を図示する。

    Args:
        Xtrain (np.ndarrray): 訓練データ説明変数
        Xtest (np.ndarray): テストデータ説明変数
        X (np.ndarray): 全説明変数
    """
    fig, ax = plt.subplots()
    ax.plot(Xtrain[:, 0], Xtrain, "o")
    ax.set_xlabel("X0")
    ax.set_ylabel("$X^{train}$")
    # xの表示範囲を合わせるため。
    ax.set_xlim((X[:, 0].min(), X[:, 0].max()))
    fig.show()

    fig, ax = plt.subplots()
    ax.plot(Xtest[:, 0], Xtest, "o")
    ax.set_xlabel("X0")
    ax.set_ylabel("$X^{test}$")
    # xの表示範囲を合わせるため。
    ax.set_xlim((X[:, 0].min(), X[:, 0].max()))
    fig.show()
    
plotX(g_Xtrain,g_Xtest,g_X)


最後に
($y^{obs}$ vs $y^{test}$)の
回帰結果を可視化します。


In [ ]:
plot_y_yp(g_ytrain, g_ytrainp, "fited by training, predicted by training")
plot_y_yp(g_ytest, g_ytestp, "fited by training, predicted by test")

### 新規データを用いた予測図


In [ ]:
g_y.shape, g_yp.shape, g_y_new.shape, g_yp_new.shape

In [ ]:
def plot_y_yp(y, yp, y_new, yp_new):
    """y ypを図示する。

    Args:
        y (np.ndarray): 観測データ目的変数値
        yp (np.ndarray): 観測データ目的変数予測値
        y_new (np.ndarray): 新規データ目的変数値
        yp_new (np.ndarray): 新規データ目的変数予測値
    """
    fig, ax = plt.subplots()
    ax.scatter(y,yp,label="obs")
    ax.scatter(y_new, yp_new, label="new")
    ax.set_xlabel("$y^{obs}$")
    ax.set_ylabel("$y^{pred}$")
    ax.legend()
    
plot_y_yp(g_y, g_yp, g_y_new, g_yp_new)

In [ ]:
def plotXy(X,y,yp, X_new,y_new, yp_new):
    """目的変数観測データと新規データの表示

    Args:
        X (np.ndarray): 観測データ説明変数値
        y (np.ndarray): 観測データ目的変数値
        X_new (np.ndarray): 新規データ説明変数値
        y_new (np.ndarray): 新規データ目的変数値
    """
    fig, ax = plt.subplots()
    ax.plot(X[:,0],y,label="obs")
    ax.plot(X_new[:,0], y_new, label="new")

    ax.plot(X[:,0],yp,label="pred,obs")
    ax.plot(X_new[:,0], yp_new, label="pred,new")

    ax.set_xlabel("x1")
    ax.set_ylabel("y")
    ax.legend()
    
plotXy(g_X,g_y, g_yp, g_X_new,g_y_new, g_yp_new)

**コメント**

もし回帰性能が良くなければ

1. $x_i$と $x_i x_j$ を説明変数としてみる
2. 次は$x_i$, $x_i x_j$ と $x_i x_j x_k$を説明変数としてみる

などと高次項を入れてみるという可能性が考えられます。

高次項を入れても回帰性能が良くない場合は

1. cos(x)など素性が異なる説明変数を入れてみる。
2. 全く異なる起源の説明変数を入れてみる。
3. Kernel回帰などの非線形回帰を行う。

などの対処法が考えられます。

## 問題1

異なるデータファイルを読み込む。
異なる説明変数を用いる。


## 問題２

scalerを変えて実行する。


## 問題3

train_test_splitのtest_sizeを変えてみる。

## 問題4

regtypeを変えて実行する。